# AlgoTrade - BTC/USD Backtesting Lab

A lightweight backtesting notebook for crypto trading strategies, built on [`vectorbt`](https://github.com/polakowo/vectorbt).

**What's inside:**
- Data loading & preparation (5 years of daily BTC/USD OHLCV from Bitstamp)
- Technical indicators: SMA, RSI, MACD, Bollinger Bands, ATR
- Two backtested strategies: **Moving Average Crossover** and **RSI Mean Reversion**
- Performance metrics, equity curves, and a side-by-side strategy comparison

> Data source: `tutorial.csv` (regenerate with `data.ipynb`, which paginates Bitstamp's OHLC API).

In [1]:
import numpy as np
import pandas as pd
import vectorbt as vbt
import datetime as dt

# Notebook-wide config
CSV_PATH = "tutorial.csv"
INIT_CASH = 10_000
FEES = 0.001       # 0.1% per trade, typical Bitstamp taker fee
FREQ = "1D"

vbt.settings.set_theme("dark")
vbt.settings["plotting"]["use_widgets"] = False  # plain static figures keep the notebook file small
vbt.settings["plotting"]["layout"]["width"] = 900
vbt.settings["plotting"]["layout"]["height"] = 450

## 1. Load & Prepare Data

Load 5 years of daily BTC/USD OHLCV data and index it by timestamp.

In [2]:
df = pd.read_csv(CSV_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")
df = df.set_index("timestamp").sort_index()

open_, high, low, close, volume = df["open"], df["high"], df["low"], df["close"], df["volume"]

print(f"Rows: {len(df)}  |  Range: {df.index.min()} → {df.index.max()}")
df.head()

Rows: 1826  |  Range: 2021-08-18 00:00:00 → 2026-08-17 00:00:00


,open,high,low,close,volume
timestamp,,,,,
2021-08-18,44631.47,46041.62,44218.73,44721.13,2247.934158
2021-08-19,44734.16,47114.99,43935.54,46764.30,2776.237685
2021-08-20,46759.29,49436.10,46645.77,49356.00,2815.551595
2021-08-21,49332.31,49833.02,48300.00,48884.34,1768.241566
2021-08-22,48883.84,49540.01,48080.17,49301.86,985.765333


## 2. Technical Indicators

Compute the indicators the strategies below are built on:

- **SMA(20) / SMA(50)** - fast/slow trend-following moving averages, tuned for daily bars
- **RSI(14)** - momentum oscillator for overbought/oversold conditions
- **MACD(12, 26, 9)** - trend + momentum
- **Bollinger Bands(20, 2σ)** - volatility bands
- **ATR(14)** - average true range, for volatility context

In [3]:
fast_ma = vbt.MA.run(close, window=20)
slow_ma = vbt.MA.run(close, window=50)
rsi = vbt.RSI.run(close, window=14)
macd = vbt.MACD.run(close, fast_window=12, slow_window=26, signal_window=9)
bb = vbt.BBANDS.run(close, window=20, alpha=2)
atr = vbt.ATR.run(high, low, close, window=14)

indicators = pd.DataFrame({
    "close": close,
    "sma_fast": fast_ma.ma,
    "sma_slow": slow_ma.ma,
    "rsi": rsi.rsi,
    "macd": macd.macd,
    "macd_signal": macd.signal,
    "bb_upper": bb.upper,
    "bb_lower": bb.lower,
    "atr": atr.atr,
})
indicators.tail()

,close,sma_fast,sma_slow,rsi,macd,macd_signal,bb_upper,bb_lower,atr
timestamp,,,,,,,,,
2026-08-13,63418.04,64008.9160,63381.1924,39.836663,-234.216410,-359.817756,65399.544687,62618.287313,1212.591293
2026-08-14,62975.97,63942.3195,63446.7866,51.601720,-212.284359,-346.052358,65395.462635,62489.176365,1195.304454
2026-08-15,63025.94,63826.6725,63506.9178,52.702088,-164.354551,-324.444480,65181.664045,62471.680955,1069.186527
2026-08-16,62831.66,63783.3695,63564.7496,42.367961,-124.614423,-289.304017,65205.756829,62360.982171,1014.289657
2026-08-17,62826.96,63732.1910,63631.8230,42.712120,-147.173397,-256.550121,65213.660027,62250.721973,904.365702


In [4]:
fig = close.rename("Close").vbt.plot(trace_kwargs=dict(line=dict(color="lightskyblue")))
fast_ma.ma.rename("SMA 20").vbt.plot(trace_kwargs=dict(line=dict(color="orange", width=1)), fig=fig)
slow_ma.ma.rename("SMA 50").vbt.plot(trace_kwargs=dict(line=dict(color="magenta", width=1)), fig=fig)
bb.upper.rename("BB Upper").vbt.plot(trace_kwargs=dict(line=dict(color="gray", width=1, dash="dot")), fig=fig)
bb.lower.rename("BB Lower").vbt.plot(trace_kwargs=dict(line=dict(color="gray", width=1, dash="dot")), fig=fig)
fig.update_layout(title="BTC/USD - Daily Price with SMA & Bollinger Bands")
fig.show()

## 3. Strategy 1 - Moving Average Crossover

**Rule:** go long when SMA(20) crosses above SMA(50) (bullish momentum); exit when it crosses back below.

A classic trend-following strategy - profits in sustained trends, whipsaws in choppy/sideways markets.

In [5]:
ma_entries = fast_ma.ma_crossed_above(slow_ma)
ma_exits = fast_ma.ma_crossed_below(slow_ma)

pf_ma = vbt.Portfolio.from_signals(
    close, ma_entries, ma_exits,
    init_cash=INIT_CASH, fees=FEES, freq=FREQ,
)

pf_ma.stats()

Start                                2021-08-18 00:00:00
End                                  2026-08-17 00:00:00
Period                                1826 days 00:00:00
Start Value                                      10000.0
End Value                                    9205.656072
Total Return [%]                               -7.943439
Benchmark Return [%]                           40.486074
Max Gross Exposure [%]                             100.0
Total Fees Paid                               402.165604
Max Drawdown [%]                               58.793403
Max Drawdown Duration                 1132 days 00:00:00
Total Trades                                          24
Total Closed Trades                                   23
Total Open Trades                                      1
Open Trade PnL                               -359.108671
Win Rate [%]                                   30.434783
Best Trade [%]                                 55.455566
Worst Trade [%]                

In [6]:
pf_ma.plot(title="MA Crossover - Portfolio Value, Trades & Drawdown").show()

## 4. Strategy 2 - RSI Mean Reversion

**Rule:** go long when RSI(14) crosses above 30 (exiting oversold); exit when RSI crosses above 70 (overbought).

A contrarian strategy - profits in range-bound/choppy markets, underperforms in strong trends.

In [7]:
RSI_LOWER, RSI_UPPER = 30, 70

rsi_entries = rsi.rsi_crossed_above(RSI_LOWER)
rsi_exits = rsi.rsi_crossed_above(RSI_UPPER)

pf_rsi = vbt.Portfolio.from_signals(
    close, rsi_entries, rsi_exits,
    init_cash=INIT_CASH, fees=FEES, freq=FREQ,
)

pf_rsi.stats()

Start                         2021-08-18 00:00:00
End                           2026-08-17 00:00:00
Period                         1826 days 00:00:00
Start Value                               10000.0
End Value                             9323.809526
Total Return [%]                        -6.761905
Benchmark Return [%]                    40.486074
Max Gross Exposure [%]                      100.0
Total Fees Paid                        401.607337
Max Drawdown [%]                        63.006806
Max Drawdown Duration          1727 days 00:00:00
Total Trades                                   22
Total Closed Trades                            22
Total Open Trades                               0
Open Trade PnL                                0.0
Win Rate [%]                            72.727273
Best Trade [%]                          26.634172
Worst Trade [%]                        -42.763735
Avg Winning Trade [%]                    7.957047
Avg Losing Trade [%]                   -17.560185


In [8]:
pf_rsi.plot(title="RSI Mean Reversion - Portfolio Value, Trades & Drawdown").show()

## 5. Strategy Comparison

Compare both strategies against each other and a buy & hold benchmark.

In [9]:
pf_hold = vbt.Portfolio.from_holding(close, init_cash=INIT_CASH, freq=FREQ)

metrics = ["total_return", "sharpe_ratio", "max_dd", "win_rate", "total_trades"]

comparison = pd.DataFrame({
    "Buy & Hold": pf_hold.stats(metrics=metrics),
    "MA Crossover": pf_ma.stats(metrics=metrics),
    "RSI Mean Reversion": pf_rsi.stats(metrics=metrics),
}).T

comparison

,Total Return [%],Sharpe Ratio,Max Drawdown [%],Win Rate [%],Total Trades
Buy & Hold,40.486074,0.391484,76.663361,NaN,1.0
MA Crossover,-7.943439,0.124116,58.793403,30.434783,24.0
RSI Mean Reversion,-6.761905,0.147140,63.006806,72.727273,22.0


In [10]:
comparison["Total Return [%]"].vbt.barplot(
    trace_kwargs=dict(marker_color=["#888888", "#f4a261", "#2a9d8f"]),
).update_layout(title="Total Return by Strategy", yaxis_title="Return [%]").show()

## 6. Takeaways & Next Steps

- This backtest spans 5 years of daily bars - enough to cover multiple bull/bear cycles, but a single asset and a single history: results can still be regime-dependent (e.g. skewed by 2021 or 2024/25 bull runs) rather than a robust edge.
- Trading fees compound over years of trades - compare `Total Fees Paid` against `Total Return` before trusting either strategy.
- Ideas to extend this notebook:
  - Parameter sweeps (e.g. grid search MA windows or RSI thresholds with `vbt.Portfolio.from_signals` on broadcast arrays)
  - Add stop-loss / take-profit (`sl_stop`, `tp_stop` in `from_signals`)
  - Walk-forward / train-test split to check for overfitting
  - Combine signals (e.g. only take MA crossover trades when RSI confirms momentum)
  - Compare against other assets or a shorter/rolling timeframe to test robustness across regimes